In [1]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd, statsmodels.api as sm

df = pd.read_csv('../data/processed/merged_dataset.csv')
spread_cols = ['spread_24','spread_48','spread_72','spread_96','spread_120','spread_144','spread_168']
X = sm.add_constant(df[spread_cols])
vif = pd.DataFrame({
    'feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif)

      feature        VIF
0       const  20.303603
1   spread_24   3.705511
2   spread_48   6.515491
3   spread_72   5.908120
4   spread_96   5.754323
5  spread_120   5.286790
6  spread_144   5.129973
7  spread_168   3.248474


In [2]:
import numpy as np, pandas as pd, statsmodels.api as sm

df = pd.read_csv('../data/processed/merged_dataset.csv')

df['log_vol'] = np.log(df['realized_vol'])
df['log_lagged_vol'] = np.log(df['lagged_vol'])
df['CDD'] = np.maximum(df['temp_24'] - 291.5, 0)
df['HDD'] = np.maximum(291.5 - df['temp_24'], 0)

def run(features, target='log_vol'):
    data = df[features + [target]].dropna()
    X = sm.add_constant(data[features]); y = data[target]
    maxlags = int(0.75 * len(data) ** (1/3))
    return sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})

controls = ['log_lagged_vol', 'season_sin', 'season_cos', 'CDD', 'HDD', 'gas_price']

m1 = run(controls)                                  # baseline
m2 = run(controls + ['spread_24'])                  # + canonical spread
m3 = run(controls + ['spread_24', 'path_length'])   # + your novel feature

for name, m in [('baseline', m1), ('+spread_24', m2), ('+path_length', m3)]:
    print(f"{name:<14} R2={m.rsquared:.4f}  AdjR2={m.rsquared_adj:.4f}")

print(m3.summary())

baseline       R2=0.3449  AdjR2=0.3425
+spread_24     R2=0.3480  AdjR2=0.3452
+path_length   R2=0.3485  AdjR2=0.3453
                            OLS Regression Results                            
Dep. Variable:                log_vol   R-squared:                       0.349
Model:                            OLS   Adj. R-squared:                  0.345
Method:                 Least Squares   F-statistic:                     111.0
Date:                Mon, 27 Jul 2026   Prob (F-statistic):          9.94e-148
Time:                        16:08:19   Log-Likelihood:                -1571.8
No. Observations:                1621   AIC:                             3162.
Df Residuals:                    1612   BIC:                             3210.
Df Model:                           8                                         
Covariance Type:                  HAC                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------

In [3]:
import numpy as np, pandas as pd, statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv('../data/processed/merged_dataset.csv')
df['log_vol'] = np.log(df['realized_vol'])
df['log_lagged_vol'] = np.log(df['lagged_vol'])
df['CDD'] = np.maximum(df['temp_24'] - 291.5, 0)
df['HDD'] = np.maximum(291.5 - df['temp_24'], 0)
data = df.dropna(subset=['log_vol','log_lagged_vol','spread_24'])

formula = 'log_vol ~ log_lagged_vol + season_sin + season_cos + CDD + HDD + gas_price + spread_24'

results = []
for q in [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]:
    m = smf.quantreg(formula, data).fit(q=q)
    results.append({'quantile': q,
                    'spread_24_coef': m.params['spread_24'],
                    'pvalue': m.pvalues['spread_24']})
print(pd.DataFrame(results).round(4))

   quantile  spread_24_coef  pvalue
0      0.10          0.2297  0.1593
1      0.25          0.1486  0.2260
2      0.50          0.3436  0.0030
3      0.75          0.2096  0.1057
4      0.90          0.4006  0.0164
5      0.95          0.5113  0.0241


In [4]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv('../data/processed/merged_dataset.csv')
df['log_vol'] = np.log(df['realized_vol'])
df['log_lagged_vol'] = np.log(df['lagged_vol'])
df['CDD'] = np.maximum(df['temp_24'] - 291.5, 0)
df['HDD'] = np.maximum(291.5 - df['temp_24'], 0)
data = df.dropna(subset=['log_vol', 'log_lagged_vol', 'spread_24']).reset_index(drop=True)

formula = 'log_vol ~ log_lagged_vol + season_sin + season_cos + CDD + HDD + gas_price + spread_24'

def bootstrap_quantile(data, formula, q, n_boot=500, seed=42):
    """Bootstrap the spread_24 coefficient at quantile q."""
    rng = np.random.default_rng(seed)
    n = len(data)
    coefs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)          # resample rows with replacement
        sample = data.iloc[idx]
        try:
            m = smf.quantreg(formula, sample).fit(q=q)
            coefs.append(m.params['spread_24'])
        except Exception:
            continue
    coefs = np.array(coefs)
    return coefs.mean(), coefs.std(), np.percentile(coefs, [2.5, 97.5])

print(f"{'q':<6}{'coef':<10}{'boot_se':<10}{'95% CI':<25}{'sig?'}")
for q in [0.50, 0.75, 0.90, 0.95]:
    mean_c, se, ci = bootstrap_quantile(data, formula, q)
    sig = "YES" if (ci[0] > 0 or ci[1] < 0) else "no"
    print(f"{q:<6}{mean_c:<10.4f}{se:<10.4f}[{ci[0]:.4f}, {ci[1]:.4f}]     {sig}")

q     coef      boot_se   95% CI                   sig?


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  w

0.5   0.3435    0.1275    [0.0826, 0.5661]     YES


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  w

0.75  0.2234    0.1245    [-0.0310, 0.4822]     no


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  w

0.9   0.3391    0.1817    [0.0156, 0.6993]     YES


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (1000) reached.
  w

0.95  0.4347    0.2837    [-0.1280, 0.9375]     no


In [5]:
import statsmodels.api as sm
from scipy.stats import chi2

controls = ['log_lagged_vol', 'season_sin', 'season_cos', 'CDD', 'HDD', 'gas_price']

spike_data = data.dropna(subset=controls + ['spike_200', 'spread_24'])

# baseline: no weather uncertainty
X0 = sm.add_constant(spike_data[controls])
m0 = sm.Logit(spike_data['spike_200'], X0).fit(disp=0)

# full: add spread_24
X1 = sm.add_constant(spike_data[controls + ['spread_24']])
m1 = sm.Logit(spike_data['spike_200'], X1).fit(disp=0)

# likelihood ratio test
lr_stat = 2 * (m1.llf - m0.llf)
p_value = 1 - chi2.cdf(lr_stat, df=1)

print(f"Baseline log-likelihood: {m0.llf:.2f}")
print(f"Full log-likelihood:     {m1.llf:.2f}")
print(f"LR statistic: {lr_stat:.3f}, p-value: {p_value:.4f}")
print(f"\nspread_24 coefficient: {m1.params['spread_24']:.4f}")
print(f"spread_24 p-value:     {m1.pvalues['spread_24']:.4f}")
print(f"odds ratio: {np.exp(m1.params['spread_24']):.3f}")

Baseline log-likelihood: -319.71
Full log-likelihood:     -319.37
LR statistic: 0.674, p-value: 0.4117

spread_24 coefficient: 0.5229
spread_24 p-value:     0.4068
odds ratio: 1.687


In [6]:
# demean both before multiplying — avoids multicollinearity with main effects
data['spread_dm'] = data['spread_24'] - data['spread_24'].mean()
data['CDD_dm'] = data['CDD'] - data['CDD'].mean()
data['spread_x_cdd'] = data['spread_dm'] * data['CDD_dm']

features = controls + ['spread_24', 'spread_x_cdd']
X = sm.add_constant(data[features])
y = data['log_vol']
maxlags = int(0.75 * len(data) ** (1/3))
m_int = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})

print(m_int.summary().tables[1])

                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.9965      0.104      9.595      0.000       0.793       1.200
log_lagged_vol     0.4115      0.027     15.096      0.000       0.358       0.465
season_sin        -0.1036      0.029     -3.586      0.000      -0.160      -0.047
season_cos        -0.1481      0.058     -2.572      0.010      -0.261      -0.035
CDD                0.0291      0.012      2.524      0.012       0.006       0.052
HDD                0.0080      0.006      1.450      0.147      -0.003       0.019
gas_price          0.0685      0.009      7.565      0.000       0.051       0.086
spread_24          0.2552      0.115      2.222      0.026       0.030       0.480
spread_x_cdd      -0.1481      0.041     -3.573      0.000      -0.229      -0.067


In [7]:
data['year'] = pd.to_datetime(data['date']).dt.year
years = sorted(data['year'].unique())

print(f"{'test_year':<12}{'coef':<10}{'n_train':<10}{'n_test'}")
results = []
for i, test_year in enumerate(years):
    if i < 2:          # need at least 2 training years
        continue
    train = data[data['year'] < test_year]
    test = data[data['year'] == test_year]
    if len(test) < 30:
        continue
    m = smf.quantreg(formula, train).fit(q=0.90)
    results.append({'test_year': test_year,
                    'coef': m.params['spread_24'],
                    'n_train': len(train),
                    'n_test': len(test)})
    print(f"{test_year:<12}{m.params['spread_24']:<10.4f}{len(train):<10}{len(test)}")

coefs = [r['coef'] for r in results]
print(f"\nAll positive: {all(c > 0 for c in coefs)}")
print(f"Mean coef: {np.mean(coefs):.4f}")

test_year   coef      n_train   n_test
2022        0.8780    440       363
2023        0.5386    803       306
2024        0.6115    1109      340
2025        0.3403    1449      172

All positive: True
Mean coef: 0.5921


In [8]:
# demeaned HDD interaction (spread_dm and CDD_dm already exist from Step 3)
data['HDD_dm'] = data['HDD'] - data['HDD'].mean()
data['spread_x_hdd'] = data['spread_dm'] * data['HDD_dm']

features = controls + ['spread_24', 'spread_x_cdd', 'spread_x_hdd']
X = sm.add_constant(data[features])
y = data['log_vol']
maxlags = int(0.75 * len(data) ** (1/3))
m_hdd = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})

print(m_hdd.summary().tables[1])

                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              1.1213      0.099     11.297      0.000       0.927       1.316
log_lagged_vol     0.3979      0.028     14.413      0.000       0.344       0.452
season_sin        -0.1072      0.029     -3.734      0.000      -0.163      -0.051
season_cos        -0.1842      0.059     -3.101      0.002      -0.301      -0.068
CDD                0.0261      0.011      2.307      0.021       0.004       0.048
HDD                0.0097      0.006      1.692      0.091      -0.002       0.021
gas_price          0.0693      0.010      7.181      0.000       0.050       0.088
spread_24          0.1121      0.115      0.971      0.331      -0.114       0.338
spread_x_cdd      -0.0202      0.051     -0.399      0.690      -0.119       0.079
spread_x_hdd       0.0624      0.018      3.519      0.000       0.028       0.097


In [9]:
# ============================================================
# ROBUSTNESS TEST 1: Cold vs mild subsample split
# ============================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv('../data/processed/merged_dataset.csv')
df['log_vol'] = np.log(df['realized_vol'])
df['log_lagged_vol'] = np.log(df['lagged_vol'])
df['CDD'] = np.maximum(df['temp_24'] - 291.5, 0)
df['HDD'] = np.maximum(291.5 - df['temp_24'], 0)
data = df.dropna(subset=['log_vol', 'log_lagged_vol', 'spread_24']).reset_index(drop=True)

controls = ['log_lagged_vol', 'season_sin', 'season_cos', 'CDD', 'HDD', 'gas_price']

def run_hac_ols(subset, features, target='log_vol'):
    d = subset[features + [target]].dropna()
    X = sm.add_constant(d[features])
    y = d[target]
    maxlags = int(0.75 * len(d) ** (1/3))
    return sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})

hdd_median = data['HDD'].median()
cold = data[data['HDD'] > hdd_median]
mild = data[data['HDD'] <= hdd_median]

m_cold = run_hac_ols(cold, controls + ['spread_24'])
m_mild = run_hac_ols(mild, controls + ['spread_24'])

print(f"COLD half (HDD > {hdd_median:.1f}), n={len(cold)}")
print(f"  spread_24 coef: {m_cold.params['spread_24']:.4f}")
print(f"  p-value:        {m_cold.pvalues['spread_24']:.4f}")
print(f"  95% CI:         [{m_cold.conf_int().loc['spread_24', 0]:.4f}, {m_cold.conf_int().loc['spread_24', 1]:.4f}]")
print()
print(f"MILD half (HDD <= {hdd_median:.1f}), n={len(mild)}")
print(f"  spread_24 coef: {m_mild.params['spread_24']:.4f}")
print(f"  p-value:        {m_mild.pvalues['spread_24']:.4f}")
print(f"  95% CI:         [{m_mild.conf_int().loc['spread_24', 0]:.4f}, {m_mild.conf_int().loc['spread_24', 1]:.4f}]")

COLD half (HDD > 5.8), n=810
  spread_24 coef: 0.6709
  p-value:        0.0000
  95% CI:         [0.3806, 0.9611]

MILD half (HDD <= 5.8), n=811
  spread_24 coef: -0.3429
  p-value:        0.0396
  95% CI:         [-0.6695, -0.0163]


In [10]:
# ============================================================
# ROBUSTNESS TEST 2: Walk-forward with HDD interaction spec
# ============================================================
data['spread_dm'] = data['spread_24'] - data['spread_24'].mean()
data['HDD_dm'] = data['HDD'] - data['HDD'].mean()
data['spread_x_hdd'] = data['spread_dm'] * data['HDD_dm']
data['year'] = pd.to_datetime(data['date']).dt.year

features_int = controls + ['spread_24', 'spread_x_hdd']
years = sorted(data['year'].unique())

print(f"{'test_year':<12}{'spread_x_hdd':<15}{'spread_24':<12}{'n_train':<10}{'n_test'}")
coefs = []
for i, test_year in enumerate(years):
    if i < 2:
        continue
    train = data[data['year'] < test_year]
    test = data[data['year'] == test_year]
    if len(test) < 30:
        continue
    m = run_hac_ols(train, features_int)
    coefs.append(m.params['spread_x_hdd'])
    print(f"{test_year:<12}{m.params['spread_x_hdd']:<15.4f}{m.params['spread_24']:<12.4f}{len(train):<10}{len(test)}")

print(f"\nspread_x_hdd all positive: {all(c > 0 for c in coefs)}")
print(f"Mean spread_x_hdd: {np.mean(coefs):.4f}")

test_year   spread_x_hdd   spread_24   n_train   n_test
2022        0.0839         0.2951      440       363
2023        0.0797         0.2827      803       306
2024        0.0693         0.1842      1109      340
2025        0.0711         0.1151      1449      172

spread_x_hdd all positive: True
Mean spread_x_hdd: 0.0760
